# End-to-End Machine Learning & Vision Analytics Framework

A modular framework spanning the core pillars of applied ML: **exploratory data analysis**, **supervised learning**, **unsupervised learning**, and **classical computer vision**, packaged as reusable, tested Python modules (`src/`) and demonstrated end-to-end in this notebook.

**Pipeline covered:**
1. Exploratory Data Analysis (EDA) — distribution & relationship plots
2. Supervised Learning — Decision Tree vs. Random Forest (+ feature selection, + Optuna tuning)
3. Unsupervised Learning — K-Means with automatic model selection (silhouette analysis) and external validation metrics
4. Computer Vision — classical image segmentation benchmarked with Dice / IoU, plus from-scratch 2D convolution for edge detection
5. (Extension) Deep Learning — a PyTorch feed-forward classifier for image data

All heavy lifting lives in `src/`; this notebook is the demo / report layer.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_digits, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

from src import visualization as viz
from src import classification as clf_module
from src import clustering as clust_module
from src import segmentation as seg_module
from src import convolution as conv_module

%matplotlib inline
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

---
## Part 1 — Exploratory Data Analysis

We start with distribution and relationship plots on the classic **Iris** dataset, using the reusable helpers in `src/visualization.py`.

In [ ]:
iris_sk = load_iris(as_frame=True)
iris_df = iris_sk.frame.rename(columns={
    "sepal length (cm)": "sepal_length", "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length", "petal width (cm)": "petal_width",
})
iris_df["species"] = iris_df["target"].map(dict(enumerate(iris_sk.target_names)))
iris_df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
viz.plot_boxplot(iris_df, x="species", y="sepal_length", ax=axes[0])
viz.plot_violin(iris_df, x="species", y="petal_length", ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
viz.plot_scatter_with_fit(iris_df, x="sepal_length", y="sepal_width", ax=ax)
plt.show()

In [ ]:
# Full pairwise correlogram, colored by species
viz.plot_pairwise(iris_df, hue="species")
plt.show()

---
## Part 2 — Supervised Learning

We compare a Decision Tree, a Random Forest, and a Random-Forest + ANOVA feature-selection pipeline on two datasets:
- **Iris** (4 features, 3 classes) — a quick sanity check
- **Digits** (64 pixel features, 10 classes) — where feature selection matters more

`src/classification.py` also exposes `tune_with_optuna(...)` for automated hyperparameter / model-family search (optional `optuna` dependency).

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE)

results_iris = clf_module.compare_classifiers(X_train, y_train, X_test, y_test)
pd.DataFrame([{"model": r.name, "test_accuracy": r.accuracy} for r in results_iris])

In [ ]:
digits = load_digits()
Xd, yd = digits.data, digits.target
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, random_state=RANDOM_STATE)

results_digits = clf_module.compare_classifiers(Xd_train, yd_train, Xd_test, yd_test)
pd.DataFrame([{"model": r.name, "test_accuracy": r.accuracy} for r in results_digits])

In [ ]:
print(results_digits[0].report)

**Optional — automated hyperparameter search with Optuna** (requires `pip install optuna`):

```python
study = clf_module.tune_with_optuna(X_train, y_train, X_test, y_test, n_trials=50)
print(study.best_trial.params, study.best_value)
```

---
## Part 3 — Unsupervised Learning: K-Means Clustering

`src/clustering.py` provides `KMeansExplorer`, which fits K-Means over a range of `k`, scores each fit with the average silhouette coefficient, and automatically selects the best `k` — removing the need to hand-pick cluster counts.

In [ ]:
X_blobs, y_blobs_true = make_blobs(
    n_samples=400, centers=4, cluster_std=[1.0, 2.0, 0.7, 1.3],
    n_features=2, random_state=7
)

explorer = clust_module.KMeansExplorer(k_range=range(2, 8))
explorer.fit(X_blobs)
print(explorer.summary())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
explorer.plot_silhouette_curve(ax=axes[0])
explorer.plot_best_clustering(ax=axes[1])
plt.tight_layout()
plt.show()

When ground-truth labels *are* available, `bench_k_means(...)` reports external validation metrics (homogeneity, completeness, V-measure, ARI, AMI) alongside internal ones (inertia, silhouette) — useful for sanity-checking clustering against a labeled dataset like Iris.

In [ ]:
bench_rows = []
for k in [2, 3, 4]:
    km = KMeans(init="k-means++", n_clusters=k, n_init=4, random_state=0)
    bench_rows.append(clust_module.bench_k_means(km, name=f"k={k}", data=X_train, labels=y_train))

pd.DataFrame(bench_rows).round(3)

---
## Part 4 — Computer Vision: Image Segmentation Benchmark

Five classical, unsupervised segmentation methods — Otsu thresholding, adaptive thresholding, K-Means intensity clustering, Canny edge detection + hole-filling, and marker-based Watershed — are run on a synthetic test image and scored against its ground-truth mask with **Dice** and **IoU**.

This benchmark harness (`src/segmentation.py`) generalizes directly to real microscopy / medical-imaging segmentation tasks.

In [ ]:
img, gt_mask = seg_module.make_synthetic_image()
results = seg_module.run_all(img)
scores = seg_module.evaluate(gt_mask, results)
pd.DataFrame(scores).round(3)

In [ ]:
seg_module.plot_results(img, gt_mask, results)
plt.show()

---
## Part 5 — From-Scratch 2D Convolution

To ground the intuition behind CNN convolutional layers, `src/convolution.py` implements 2D convolution directly with NumPy (no `nn.Conv2d`), applied here as an edge-detection kernel over the segmentation benchmark image.

In [ ]:
edge_output = conv_module.convolve2d(img.astype(float), conv_module.KERNELS["edge_detect"], padding=2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(edge_output, cmap="gray"); axes[1].set_title("Edge-detect kernel (from-scratch conv2D)"); axes[1].axis("off")
plt.tight_layout()
plt.show()

---
## Part 6 — Extension: Deep Learning Classifier (PyTorch)

`src/vision_cnn.py` wraps a small fully-connected PyTorch network (28x28 input, two hidden layers, 10-class output) plus train/eval loop utilities, so the framework has a clear on-ramp from classical ML into deep learning. Requires the optional `torch` / `torchvision` dependencies.

```python
from src import vision_cnn

model, device = vision_cnn.build_model()
train_loader, test_loader = vision_cnn.get_fashion_mnist_loaders()

import torch
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

for epoch in range(5):
    train_loss = vision_cnn.train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_loss, test_acc = vision_cnn.evaluate(model, test_loader, loss_fn, device)
    print(f"epoch {epoch}: train_loss={train_loss:.3f} test_acc={test_acc:.3f}")
```

---
## Summary

| Module | Technique | What it demonstrates |
|---|---|---|
| `visualization.py` | EDA (box/violin/scatter/pairplot) | Data exploration & communication |
| `classification.py` | Decision Tree / Random Forest / feature selection / Optuna | Supervised learning, model comparison, hyperparameter search |
| `clustering.py` | K-Means + automatic k-selection | Unsupervised learning, model selection via silhouette analysis |
| `segmentation.py` | Otsu / Adaptive / K-Means / Canny / Watershed | Classical computer vision, quantitative benchmarking (Dice/IoU) |
| `convolution.py` | From-scratch 2D convolution | Low-level understanding of CNN mechanics |
| `vision_cnn.py` | PyTorch feed-forward classifier | Deep learning extension point |

See the metric tables and plots produced by each section above for concrete numbers on this run. This project demonstrates a full applied-ML skill set — EDA, classical ML, unsupervised learning, and computer vision — organized as a clean, reusable, testable codebase rather than one-off notebook scripts.